## Cross-Encoder SFT Pipeline

Configured to run from anywhere with absolute paths based on user environment.

In [ ]:
import json
import random
from pathlib import Path
import sys

import torch
from datasets import Dataset
from sentence_transformers import CrossEncoder
from sentence_transformers.cross_encoder import (
    CrossEncoderTrainingArguments,
    CrossEncoderTrainer,
)
from sentence_transformers.cross_encoder.losses import BinaryCrossEntropyLoss
from sentence_transformers.cross_encoder.evaluation import CrossEncoderRerankingEvaluator

# Force project root
PROJECT_ROOT = Path("/Users/guest2/Desktop/repos/talentlens/TalentLens_Public")
sys.path.insert(0, str(PROJECT_ROOT))

from streamlit.config import SKILL_SUGGESTIONS
from streamlit.job_description import SKILL_ALIASES


In [ ]:
print("1. Loading Existing Artifacts")

DATA_DIR = PROJECT_ROOT / "data" / "processed"

with open(DATA_DIR / "resumes_parsed.json", encoding="utf-8") as f:
    resumes_parsed = json.load(f)

with open(DATA_DIR / "resume_chunks.json", encoding="utf-8") as f:
    resume_chunks = json.load(f)

print(f"Loaded {len(resumes_parsed)} parsed resumes and {len(resume_chunks)} chunks.")


In [ ]:
print("1.5 Loading training pool texts")
train_resume_texts = []
train_texts_path = PROJECT_ROOT / "data" / "processed" / "train_resume_texts.json"

if train_texts_path.exists():
    import json
    with open(train_texts_path, "r", encoding="utf-8") as f:
        train_resume_texts = json.load(f)
    print(f"Loaded {len(train_resume_texts)} train texts from cache.")
else:
    import json
    import pdfplumber
    import glob
    
    print("Extracting train side PDFs...")
    train_pdfs = glob.glob(str(PROJECT_ROOT / "data" / "train" / "*.pdf"))
    
    for i, pdf_path in enumerate(train_pdfs):
        if i % 100 == 0:
            print(f"Processing train PDF {i}/{len(train_pdfs)}...")
        try:
            with pdfplumber.open(pdf_path) as pdf:
                text = " ".join(page.extract_text() for page in pdf.pages if page.extract_text())
                if len(text.split()) > 80:
                    train_resume_texts.append({"filename": Path(pdf_path).name, "text": text})
        except Exception:
            pass
            
    with open(train_texts_path, "w", encoding="utf-8") as f:
        json.dump(train_resume_texts, f)
        
    print(f"Extracted {len(train_resume_texts)} useful texts from train PDFs.")

print("Added train texts to negative sampling pool.")


In [ ]:
print("3. Train/Validation Split")

random.shuffle(pairs)
split_idx = int(len(pairs) * 0.8)
train_pairs = pairs[:split_idx]
val_pairs = pairs[split_idx:]

print(f"Train size: {len(train_pairs)}, Validation size: {len(val_pairs)}")

train_dataset = Dataset.from_list(train_pairs)
val_dataset = Dataset.from_list(val_pairs)


In [ ]:
print("4. Building evaluator samples")

eval_samples = []
for pair in val_pairs:
    if pair["label"] == 1.0:
        eval_samples.append(
            {"query": pair["query"], "positive": [pair["passage"]], "negative": []}
        )
    else:
        eval_samples.append(
            {"query": pair["query"], "positive": [], "negative": [pair["passage"]]}
        )

evaluator = CrossEncoderRerankingEvaluator(eval_samples, name="talentlens-eval")


In [ ]:
print("5. Supervised Fine-Tuning setup")

model_name = "cross-encoder/ms-marco-MiniLM-L6-v2"
model = CrossEncoder(model_name, num_labels=1, max_length=512)

loss = BinaryCrossEntropyLoss(model)

output_dir = str(PROJECT_ROOT / "models" / "talentlens-cross-encoder-sft-v1")

args = CrossEncoderTrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=10,
)

trainer = CrossEncoderTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    loss=loss,
    evaluator=evaluator,
)


In [ ]:
print("6. Training...")
trainer.train()

print("7. Saving model...")
model.save_pretrained(output_dir)
print(f"Model saved to {output_dir}")
